# Anatomía del sistema, paso a paso — con un mundo de juguete

Este notebook recorre **toda la cadena** del sistema, celda por celda. Las partes 1-8 no llaman
a ningún LLM (maquinaria determinística, corre gratis en segundos); la parte 9 es el episodio
end-to-end con un agente real (~US$0.05):

1. **El mundo** — la verdad oculta y sus trampas
2. **Los regímenes** — observar vs intervenir (`do()`)
3. **El canal** — las fuentes corrompidas que ve el agente
4. **El presupuesto** — la presión que hace aparecer los vicios
5. **Dos investigadores** — el seducido y el cuidadoso (scriptados)
6. **La entrega** — el contrato `model(regime, n, seed)`
7. **La batería y la nota** — energy distance, anclas, D_MAX y el R final
8. **Producción real** — el caso `first_story_scarce_v0` con el server del repo
9. **El agente LLM real** — el episodio completo con traza turno por turno, la nota y la disección

El mundo de juguete (**la cafetería**) es un gemelo en miniatura del mundo real del repo
(`first_story_scarce_v0`): misma estructura de trampa, nombres distintos, números redondos.
Lo que entiendas acá calza 1:1 con el caso de producción del final.

## Parte 0 — Setup

Solo imports del repo (nada se modifica; el notebook consume las piezas tal cual existen).

In [1]:
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                            # el sandbox de submissions se lanza desde la raiz
sys.path.insert(0, str(ROOT))             # el paquete wager
sys.path.insert(0, str(ROOT / "scripts")) # las piezas del probe (manifests, referencias)
print("repo:", ROOT)

repo: C:\Users\YT40432\Desktop\lp\research\lucaspecina\research-worlds-envs


## Parte 1 — El mundo: la verdad oculta

Un **mundo** es un programa que conoce la verdad completa y genera datos bajo cualquier régimen.
El agente jamás ve este código — es la "hoja de respuestas" del examinador.

**La cafetería**: los cafés salen cada vez peores y el folklore del local culpa al **molinillo**
("comprá uno premium"). La verdad: el **cansancio del barista** — que nunca se registró en ningún
log — se acumula a lo largo del día y empeora *dos* cosas a la vez: la lectura de molienda y el
sabor. La correlación molienda↔sabor que todos ven es real pero **no es causal**: las dos variables
son hijas del mismo padre oculto (un *confounder*).

- `molinillo` — perilla real sobre la molienda... pero **no toca el sabor** (la trampa)
- `descansos` — perilla que sí importa (controla el cansancio)
- `hora_norm` — el timestamp, la única huella del confounder en los datos

In [2]:
def toy_sample(regime, n, seed):
    """La VERDAD de la cafeteria. server-side: el agente nunca lee esto."""
    rng = np.random.default_rng(seed)
    cfg = regime.config

    if "descansos" in cfg:                       # regimen intervenido: descansos fijados
        cansancio = 10.0 - float(cfg["descansos"]) + rng.normal(0, 0.3, n)
        hora = np.zeros(n)
    else:                                        # historico: el dia transcurre
        hora = rng.uniform(0, 1, n)
        cansancio = 2.0 + 6.0 * hora + rng.normal(0, 0.3, n)

    if "molinillo" in cfg:
        molinillo = np.full(n, float(cfg["molinillo"]))
    else:
        molinillo = np.clip(rng.normal(5, 1, n), 0, 10)

    molienda = 6 + 1.0 * (molinillo - 5) - 0.8 * (cansancio - 5) + rng.normal(0, 0.5, n)
    sabor    = 30 - 2.0 * (cansancio - 5) + rng.normal(0, 1.5, n)   # el molinillo NO entra
    return pd.DataFrame({"hora_norm": hora, "molienda": molienda, "sabor": sabor})

def regime(config=None):
    """El contrato real de regimen: config (perillas), context, horizon."""
    return SimpleNamespace(config=dict(config or {}), context={}, horizon=None)

toy_sample(regime(), 5, seed=1)

,hora_norm,molienda,sabor
0,0.511822,6.163972,29.602529
1,0.950464,4.533217,24.503203
2,0.144160,6.685883,35.862510
3,0.948649,3.205121,25.907550
4,0.311831,6.221902,28.014799


## Parte 2 — Regímenes: observar vs intervenir

El **régimen** es el objeto que le dice al mundo bajo qué condiciones generar. `config` vacío =
el proceso histórico tal como ocurrió. `config` con perillas = una **intervención** (`do()`):
el mundo fija esa perilla y el resto sigue su curso natural.

Mirá la diferencia entre "los cafés con molienda alta salieron ricos" (observación) y
"forzar la molienda alta NO mejora el sabor" (intervención):

In [3]:
hist = toy_sample(regime(), 4000, seed=7)

print("HISTORICO  — correlacion molienda-sabor:", round(hist["molienda"].corr(hist["sabor"]), 3))
print("           (la correlacion seductora: parece que la molienda importa)\n")

for cfg in [{"molinillo": 0.0}, {"molinillo": 10.0}, {"descansos": 8.0}, {"descansos": 2.0}]:
    df = toy_sample(regime(cfg), 4000, seed=7)
    print(f"do({cfg}) -> sabor medio = {df['sabor'].mean():6.2f}")

HISTORICO  — correlacion molienda-sabor: 0.717
           (la correlacion seductora: parece que la molienda importa)

do({'molinillo': 0.0}) -> sabor medio =  30.02
do({'molinillo': 10.0}) -> sabor medio =  30.02
do({'descansos': 8.0}) -> sabor medio =  36.03
do({'descansos': 2.0}) -> sabor medio =  24.03


**La trampa, en números**: el histórico muestra una correlación fuerte, pero mover el
molinillo de 0 a 10 no mueve el sabor ni medio punto — mientras que mover los descansos lo mueve
~12 puntos. El folklore compraría un molinillo carísimo que no cambia nada.

En el repo real esto es idéntico: `feedstock_grade` (el proveedor, folklore) vs `humidity`
(el hall, la causa) en [cases/first_story_scarce_v0/world.py](../cases/first_story_scarce_v0/world.py).

## Parte 3 — El canal: lo que el agente VE

El agente nunca toca `toy_sample` directo. Compra **vistas corrompidas**:
- el **canal de medición** agrega ruido de instrumento a las lecturas,
- las **columnas del confounder se ocultan** (el cansancio jamás se logueó),
- cada fila **cuesta** presupuesto.

Acá está la vista "registros históricos" armada a mano (en producción esto lo empaqueta el
server leyendo la declaración del caso — la vemos al final):

In [4]:
RUIDO_INSTRUMENTO = 1.0   # el sensor de sabor tiene error propio
PRECIO_POR_FILA   = 0.5

def vista_registros(n, seed):
    """Lo que ve el agente: sabor CON error de medicion, cansancio OCULTO."""
    limpio = toy_sample(regime(), n, seed)
    rng = np.random.default_rng(seed + 999)
    vista = limpio.copy()
    vista["sabor"] = vista["sabor"] + rng.normal(0, RUIDO_INSTRUMENTO, n)  # el canal
    return vista[["hora_norm", "molienda", "sabor"]], n * PRECIO_POR_FILA  # sin 'cansancio'

muestra, costo = vista_registros(200, seed=11)
print(f"200 filas costaron {costo} de presupuesto")
muestra.head()

200 filas costaron 100.0 de presupuesto


,hora_norm,molienda,sabor
0,0.128570,8.935924,35.498312
1,0.499278,7.193964,27.388597
2,0.601498,6.071127,28.895390
3,0.028689,8.888484,32.203058
4,0.147926,8.753545,37.710857


Así se declara lo mismo en el caso real — el server lo lee de `meta.json` y arma las
fuentes solo (`channel` = el ruido del instrumento, `hidden_columns` = lo que jamás se logueó,
`cost_per_row` = el precio):

In [5]:
import json
meta = json.loads((ROOT / "cases/first_story_scarce_v0/meta.json").read_text(encoding="utf-8"))
print(json.dumps(meta["episode"]["observe_sources"], indent=2)[:800])

{
  "registros_linea": {
    "cost_per_row": 0.5,
    "config": {},
    "context": {},
    "channel": {
      "column": "outcome",
      "noise_sd": 1.5,
      "replicates": 1
    },
    "hidden_columns": [
      "ambient"
    ]
  },
  "replicas_calibracion": {
    "cost_per_row": 15.0,
    "config": {},
    "context": {},
    "channel": {
      "column": "outcome",
      "noise_sd": 1.5,
      "replicates": 2
    },
    "hidden_columns": [
      "ambient"
    ]
  }
}


## Parte 4 — El presupuesto: la presión

El episodio da un presupuesto finito. Observar es barato; **experimentar es caro** (costo fijo +
por fila). Esa aritmética es el corazón del examen: obliga a *decidir* qué evidencia comprar.
Un mundo con todo gratis no mide juicio (nadie tiene que elegir); uno carísimo tampoco (nadie
puede investigar). La escasez está calibrada para que la jugada viciosa sea la tentadora.

In [6]:
PRESUPUESTO = 5000.0
EXP_FIJO, EXP_POR_FILA = 100.0, 2.0

gasto = 0.0
_, c = vista_registros(400, seed=3); gasto += c            # 400 filas de registros
gasto += EXP_FIJO + EXP_POR_FILA * 50                       # un experimento de 50 unidades
gasto += EXP_FIJO + EXP_POR_FILA * 50                       # otro
print(f"gastado: {gasto}  |  queda: {PRESUPUESTO - gasto}")
print("(cada experimento de 50 filas cuesta lo que 400 filas historicas)")

gastado: 600.0  |  queda: 4400.0
(cada experimento de 50 filas cuesta lo que 400 filas historicas)


## Parte 5 — Dos investigadores (scriptados, sin LLM)

Los dos ven exactamente los mismos datos. La diferencia es **el juicio**.

**El SEDUCIDO**: mira el histórico, corre una regresión, encuentra el coeficiente y se casa
con la primera historia — la que el folklore ya le había susurrado.

In [7]:
datos, _ = vista_registros(400, seed=3)

# regresion sabor ~ molienda (la historia facil)
X = np.column_stack([np.ones(len(datos)), datos["molienda"]])
beta = np.linalg.lstsq(X, datos["sabor"].values, rcond=None)[0]
print(f"EL SEDUCIDO concluye: sabor = {beta[0]:.1f} + {beta[1]:.2f} * molienda")
print("  'cada punto de molienda vale", round(beta[1], 2), "de sabor — cambien el molinillo!'")

EL SEDUCIDO concluye: sabor = 21.2 + 1.44 * molienda
  'cada punto de molienda vale 1.44 de sabor — cambien el molinillo!'


**El CUIDADOSO**: nota que hay timestamps, **estratifica por hora** (la escapatoria barata:
dentro de una ventana angosta de tiempo, el confounder casi no varía) y ve que la correlación se
evapora. Después paga **un experimento** (la escapatoria cara pero definitiva) para confirmar.

In [8]:
# escapatoria barata: la correlacion DENTRO de cada franja horaria
for lo, hi in [(0.0, 0.2), (0.4, 0.6), (0.8, 1.0)]:
    franja = datos[(datos.hora_norm >= lo) & (datos.hora_norm < hi)]
    print(f"franja hora {lo}-{hi}: corr molienda-sabor = {franja['molienda'].corr(franja['sabor']):+.3f}  (n={len(franja)})")

# escapatoria cara: el experimento do(molinillo)
exp_lo = toy_sample(regime({"molinillo": 0.0}), 50, seed=21)
exp_hi = toy_sample(regime({"molinillo": 10.0}), 50, seed=22)
print(f"\nexperimento: sabor con molinillo=0 -> {exp_lo['sabor'].mean():.2f}   molinillo=10 -> {exp_hi['sabor'].mean():.2f}")
print("EL CUIDADOSO concluye: el molinillo NO mueve el sabor; hay un factor del dia (oculto) que mueve ambos.")

franja hora 0.0-0.2: corr molienda-sabor = -0.034  (n=73)
franja hora 0.4-0.6: corr molienda-sabor = +0.105  (n=83)
franja hora 0.8-1.0: corr molienda-sabor = +0.391  (n=74)

experimento: sabor con molinillo=0 -> 30.10   molinillo=10 -> 29.81
EL CUIDADOSO concluye: el molinillo NO mueve el sabor; hay un factor del dia (oculto) que mueve ambos.


## Parte 6 — La entrega: el contrato

El agente no entrega un ensayo: entrega **un programa** con la misma firma que el mundo —
`model(regime, n, seed) -> tabla` — su *maqueta* del proceso. Todo lo que crea del mundo tiene
que estar ahí adentro, ejecutable. (El contrato exacto y sin ambigüedades es el `DeliverySpec v1`
de [scripts/out/probe_0135/DELIVERY_SPEC_v1.md](../scripts/out/probe_0135/DELIVERY_SPEC_v1.md).)

Las dos entregas, tal como las escribirían nuestros investigadores:

In [9]:
modelo_seducido = '''
import numpy as np, pandas as pd
def model(regime, n, seed):
    rng = np.random.default_rng(seed)
    if "molinillo" in regime.config:                      # su unica palanca: el molinillo
        molienda = float(regime.config["molinillo"]) - 5 + 6 + rng.normal(0, 0.55, n)
    else:
        molienda = rng.normal(6.0, 1.9, n)                # reproduce el pasado que ajusto
    sabor = 21.2 + 1.44*molienda + rng.normal(0, 2.4, n)  # su regresion, tomada como CAUSA
    return pd.DataFrame({"molienda": molienda, "sabor": sabor})
'''

modelo_cuidadoso = '''
import numpy as np, pandas as pd
def model(regime, n, seed):
    rng = np.random.default_rng(seed)
    cfg = regime.config
    if "descansos" in cfg:
        cansancio = 10.0 - float(cfg["descansos"]) + rng.normal(0, 0.3, n)
    else:
        cansancio = 2.0 + 6.0*rng.uniform(0, 1, n) + rng.normal(0, 0.3, n)
    if "molinillo" in cfg:
        molinillo = np.full(n, float(cfg["molinillo"]))
    else:
        molinillo = np.clip(rng.normal(5, 1, n), 0, 10)
    molienda = 6 + 1.0*(molinillo-5) - 0.8*(cansancio-5) + rng.normal(0, 0.5, n)
    sabor    = 30 - 2.0*(cansancio-5) + rng.normal(0, 1.5, n)
    return pd.DataFrame({"molienda": molienda, "sabor": sabor})
'''
print("dos entregas listas (strings de codigo, igual que entregan los agentes reales)")

dos entregas listas (strings de codigo, igual que entregan los agentes reales)


## Parte 7 — La batería, la distancia y la nota

**La batería** es la hoja de preguntas del examen: una lista de regímenes con pesos, **oculta
para el agente**. Incluye el histórico Y las intervenciones — ahí es donde el seducido no puede
esconderse: su modelo predice que `do(molinillo=10)` mejora el sabor, y el mundo verdadero dice
que no.

**La nota**: por cada régimen se comparan las distribuciones (verdad vs maqueta) con una
distancia estadística (*energy distance* — la pieza real del repo). Se normaliza contra un
tope `D_MAX` = 1.5× la distancia de un modelo "que no sabe nada" (el ancla): quedar peor que
no-saber-nada da 0. **Cero LLM en todo este camino** — regla dura del proyecto.

In [10]:
from wager.reward.distance import TruthSide          # la pieza REAL del scoring
from wager.reward.sandbox import SandboxedSubmission  # el sandbox REAL de submissions

COLS = ["molienda", "sabor"]
BATERIA = [  # (peso, regimen) — la hoja de preguntas del examinador
    (1.0, regime()),                      # el proceso historico
    (2.0, regime({"molinillo": 10.0})),   # el counterfactual del folklore (la pregunta del cliente!)
    (2.0, regime({"molinillo": 0.0})),
    (2.0, regime({"descansos": 8.0})),    # la palanca verdadera
    (1.0, regime({"molinillo": 8.0, "descansos": 3.0})),
]
N = 3000

# el ANCLA: un "modelo" que no sabe nada de perillas — la mezcla de todos los regimenes.
# Quedar peor que el ancla da 0. D_MAX = 1.5 x distancia(ancla, verdad), como en produccion.
verdades = [toy_sample(ns, N, seed=100 + i)[COLS] for i, (_, ns) in enumerate(BATERIA)]
pool = pd.concat(verdades, ignore_index=True)

def nota_R(codigo):
    filas, total_peso, total = [], 0.0, 0.0
    rng = np.random.default_rng(999)
    with SandboxedSubmission(codigo, COLS, timeout_s=15.0) as sb:
        for i, (peso, ns) in enumerate(BATERIA):
            ts = TruthSide(verdades[i], COLS)
            d = ts.distance_to(sb.run(ns, N, 200 + i))
            ancla = pool.sample(N, random_state=int(rng.integers(1e6))).reset_index(drop=True)
            d_max = 1.5 * ts.distance_to(ancla)
            item = 1.0 - min(d, d_max) / d_max               # 1 = clavado, 0 = peor que el ancla
            filas.append((ns.config, round(d, 3), round(item, 3)))
            total += peso * item; total_peso += peso
    return total / total_peso, filas

for nombre, codigo in [("SEDUCIDO", modelo_seducido), ("CUIDADOSO", modelo_cuidadoso)]:
    R, filas = nota_R(codigo)
    print(f"\n{nombre}: R = {R:.3f}")
    for cfg, d, item in filas:
        print(f"   regimen {str(cfg):38} distancia={d:7.3f}  item={item:.3f}")


SEDUCIDO: R = 0.414
   regimen {}                                     distancia=  0.009  item=0.980
   regimen {'molinillo': 10.0}                    distancia=  1.884  item=0.406
   regimen {'molinillo': 0.0}                     distancia=  1.928  item=0.677
   regimen {'descansos': 8.0}                     distancia=  4.565  item=0.083
   regimen {'molinillo': 8.0, 'descansos': 3.0}   distancia=  7.977  item=0.000



CUIDADOSO: R = 1.000
   regimen {}                                     distancia=  0.001  item=0.998
   regimen {'molinillo': 10.0}                    distancia=  0.001  item=1.000
   regimen {'molinillo': 0.0}                     distancia=  0.001  item=1.000
   regimen {'descansos': 8.0}                     distancia=  0.001  item=1.000
   regimen {'molinillo': 8.0, 'descansos': 3.0}   distancia=  0.001  item=1.000


**Leé la tabla**: el seducido clava el histórico (ahí su historia ajusta perfecto — por eso
es seductora) y se **hunde exactamente en los regímenes intervenidos**. El vicio no se castiga
por decreto: es la jugada perdedora del mundo. Eso es "las conductas se observan, las
consecuencias se cobran" — el principio central del scoring.

*(Nota fina: esta batería de juguete incluye a propósito los regímenes que cobran el defecto.
Lo que aprendimos esta semana — el audit de la entrega 0.9866 — es que las baterías reales
necesitan exactamente esto: regímenes que cobren los defectos frecuentes, y una normalización
que no los comprima.)*

## Parte 8 — Producción real: el caso del repo, de punta a punta

Todo lo anterior, empaquetado: `build_world_server` levanta el caso real
(`first_story_scarce_v0`), con su brief, sus fuentes declaradas, su presupuesto, su batería
certificada y su scoring de producción. Vas a ver el mismo flujo: describir → comprar →
experimentar → entregar → nota.

In [11]:
from wager.harness.case_episode import build_world_server

server = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=123)
ficha = server.describe()
print(ficha["brief"][:900])

# Where the money goes: supplier or hall (reject pricing)

You advise a line whose rejects have been rising for months. The plant is
convinced it knows why: **"the feedstock got worse -- switch to the premium
supplier."** That is the folklore; you are paid to check it, not to repeat it.

Your job is to hand back a model that predicts the distribution of intake
reading and final quality -- `feedstock`, `outcome` -- under any setting of
the knobs the line can actually turn: the **supplier grade** it contracts
(`feedstock_grade`), and the hall conditions it could invest in
(`humidity`, `temp`, `line_speed`).

## What is at stake

Rejects are the cost driver: **a unit finishing below the acceptance floor of
25 is rejected at the line's cost**. The line is about to spend real money on
ONE fix. Your model prices both options: what happens to the reject rate if
the premium supplier is contracte


In [12]:
obs = server.observe("registros_linea", 100)     # compra real: 100 filas historicas
print(f"quedan {server.budget_remaining:.0f} de presupuesto tras comprar 100 filas")
obs.head()

quedan 4950 de presupuesto tras comprar 100 filas


,t,feedstock,outcome
0,0.488302,5.640594,26.317199
1,0.733307,4.632230,26.606265
2,0.376352,7.024565,33.189969
3,0.906943,5.030938,25.167293
4,0.980206,3.572969,27.932368


In [13]:
from wager.contracts import ExperimentDesign

exp = server.experiment(ExperimentDesign(config={"humidity": 2.0}, n=40))
print(f"experimento do(humidity=2): sabor... digo, outcome medio = {exp['outcome'].mean():.2f}")
print(f"quedan {server.budget_remaining:.0f} de presupuesto")

experimento do(humidity=2): sabor... digo, outcome medio = 37.42
quedan 4770 de presupuesto


Y las dos entregas de verdad — la **traducción perfecta de la verdad** (la referencia del
probe de esta semana) y el **folklore** (la creencia de la planta). El server puntúa con el
camino de producción completo (batería certificada, anclas, funcionales, sandbox):

In [14]:
from probe_0135_build import reference_code, TRUTH_MANIFEST
from exp_probe_0135_stage2 import REF_FOLKLORE

server_a = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=123)
server_a.submit(reference_code(TRUTH_MANIFEST))
print(f"entrega VERDAD-EXACTA : R = {server_a.result['R']:.4f}")

server_b = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=123)
server_b.submit(REF_FOLKLORE)
print(f"entrega FOLKLORE      : R = {server_b.result['R']:.4f}")

entrega VERDAD-EXACTA : R = 1.0000


entrega FOLKLORE      : R = 0.1453


## Parte 9 — El agente LLM real, de punta a punta

Ahora sí: **el episodio completo con un agente de verdad**. Al ejecutar la celda de abajo,
DeepSeek recibe el brief (nada más — jamás ve la verdad), y juega el episodio real: escribe
celdas de Python contra `env` (comprar datos, experimentar, analizar), el kernel las ejecuta
contra el server, cada compra le cuesta presupuesto, y termina entregando su modelo con
`env.submit(...)`. Vas a ver la **traza completa turno por turno**: su razonamiento, el código
que escribió, lo que el kernel le devolvió, y el presupuesto bajando. En el turno 4 aparece
además el **evento sorpresa** del mundo (facilities encuentra el log del HVAC — la evidencia
que destraba la historia verdadera, si el agente la sabe usar).

- Costo: ~US$0.05 por episodio (10-15 llamadas cortas a DeepSeek vía Azure).
- Requiere las credenciales de Azure en el ambiente (las mismas de siempre).
- Cada corrida es distinta (el modelo tira dados) — correlo más de una vez y vas a ver la
  varianza de la que tanto hablamos, en vivo.

In [15]:
from wager.agent.cells import extract_cell
from wager.agent.llm_client import FoundryChat
from wager.harness.episode import MAX_COMPLETION_TOKENS, SYSTEM
from wager.harness.kernel_proc import KernelClient

MODELO_AGENTE = "DeepSeek-V3.2"
MAX_TURNOS = 14
REC = lambda s, k=900: (s[:k] + " [...]") if s and len(s) > k else (s or "")

def episodio_llm(seed_offset=777, verbose=True):
    server = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=seed_offset)
    chat = FoundryChat(system=SYSTEM, model=MODELO_AGENTE, max_completion_tokens=MAX_COMPLETION_TOKENS)
    ficha = server.describe()
    prompt = ("Here is the brief:\n\n" + ficha["brief"]
              + "\n\nMachine-readable sheet:\n"
              + json.dumps({k: v for k, v in ficha.items() if k != "brief"}, indent=2)
              + "\n\nReason briefly about your opening plan, then write your first cell. "
                "`env` is already in the namespace.")
    with KernelClient(server, cell_timeout_s=30.0) as kernel:
        for turno in range(1, MAX_TURNOS + 1):
            avisos = server.begin_turn(turno)
            if avisos:
                prompt = "\n".join(f"[NOTICE] {a}" for a in avisos) + "\n\n" + prompt
                if verbose: print(f"\n{'!'*70}\n EVENTO DEL MUNDO: {avisos[0][:200]}\n{'!'*70}")
            respuesta = chat.ask(prompt)
            celda = extract_cell(respuesta.content)
            if verbose:
                print(f"\n{'='*70}\n TURNO {turno}  |  presupuesto: {server.budget_remaining:.0f}\n{'='*70}")
                print("--- razonamiento del agente ---")
                print(REC(respuesta.content.split('```')[0].strip()))
                print("--- celda que escribio ---")
                print(REC(celda or "(no escribio celda)", 700))
            if celda is None:
                prompt = "Your reply contained no python cell. Write your next cell."
                continue
            resultado = kernel.run_cell(celda)
            if server.terminal:
                if verbose: print("\n>>> ENTREGO. El server puntua en silencio (el agente jamas ve esto):")
                break
            if verbose:
                print("--- respuesta del kernel ---")
                print(REC(resultado.stdout or "(sin stdout)", 600))
                if resultado.error: print("TRACEBACK:", REC(resultado.error, 300))
            prompt = (f"Kernel output (ok={resultado.ok}, budget remaining={server.budget_remaining:.0f}):\n"
                      + (resultado.stdout or "(no stdout)")
                      + ("\nTRACEBACK:\n" + resultado.error if resultado.error else "")
                      + "\n\nReason about what this result tells you (does it confirm or refute your "
                        "current hypothesis? what does it imply for the next step?), then write your "
                        "next cell (or build and env.submit(code) when your reasoning has converged).")
    return server, chat

server_ep, chat_ep = episodio_llm(seed_offset=777)   # <- ejecutala mas de una vez (cambia el seed_offset)


 TURNO 1  |  presupuesto: 5000
--- razonamiento del agente ---
Reasoning: The core question is whether the observed correlation between feedstock intake reading and final outcome is causal, driven by the supplier grade, or confounded by time-varying hall conditions that weren't logged. The data provides timestamps, which might serve as a proxy for unobserved hall conditions if those conditions drift over time. My opening plan is to first explore the historical records cheaply: check the raw correlation between feedstock and outcome, and then see if there's a time trend that could indicate changing hall conditions. This will help me decide whether to pursue confounding via time. I'll start with a small observational sample to get a sense.
--- celda que escribio ---
df_hist = env.observe('registros_linea', n=200)
print(df_hist.head())
print(df_hist.shape)
print("Correlation:", df_hist['feedstock'].corr(df_hist['outcome']))
--- respuesta del kernel ---
          t  feedstock    outcome
0


 TURNO 2  |  presupuesto: 4900
--- razonamiento del agente ---
Reasoning: The correlation between feedstock and outcome is positive and fairly strong (0.66), consistent with the plant's folklore that better feedstock leads to better outcomes. However, the timestamp is present and could be a confounder if hall conditions drift over time. I need to see if there's a time trend in either feedstock or outcome, and whether the correlation holds within narrow time slices (to control for time-varying confounders). Let's examine time trends and compute rolling correlations or split by time quantiles.
--- celda que escribio ---
import pandas as pd

# Check time trends
print("Mean feedstock over time (binned):")
df_hist['t_bin'] = pd.cut(df_hist['t'], bins=10)
print(df_hist.groupby('t_bin')[['feedstock', 'outcome']].mean())

# Compute correlation within each time bin
corrs = df_hist.groupby('t_bin').apply(lambda g: g['feedstock'].corr(g['outcome']))
print("\nCorrelation within time bins:")
print


 TURNO 3  |  presupuesto: 4900
--- razonamiento del agente ---
Reasoning: This is striking. Over time, both feedstock and outcome decline monotonically, suggesting a strong time trend (likely deteriorating hall conditions or instrument drift). The within-bin correlations are much lower (average ~0.2) and some are even negative, indicating that the overall positive correlation is largely driven by the common time trend—a classic confounding pattern. This supports the hypothesis that the observed feedstock-outcome relationship is not causal; instead, both are affected by unobserved time-varying factors (like hall conditions). The next step is to understand the measurement error structure: the instrument might have error, and we have a calibration dataset with replicates. I'll examine the calibration data to estimate instrument noise and see if there's evidence of systematic drift in measurement error over time.
--- celda que escribio ---
df_cal = env.observe('replicas_calibracion', n=10


 TURNO 4  |  presupuesto: 3400
--- razonamiento del agente ---
Reasoning: The calibration data shows high correlation between replicates (0.94) and a small mean difference with moderate noise (std ~2.05), indicating instrument error but not huge. The diff doesn't correlate strongly with time, suggesting measurement error is not systematically drifting. However, a new source `log_hvac` (hall HVAC log) has been added! That's crucial: it likely contains humidity, temp, or similar hall conditions over time. This could be the missing confounder. Before spending on experiments, I should examine this new source to see if hall conditions correlate with time and with feedstock/outcome. Let's first inspect its schema via env.describe(), then sample some data.
--- celda que escribio ---
info = env.describe()
print("Available sources:", list(info['sources'].keys()))
# Assuming log_hvac exists, let's observe a small sample
df_hvac = env.observe('log_hvac', n=50)
print(df_hvac.head())
print(df_hvac


 TURNO 5  |  presupuesto: 3375
--- razonamiento del agente ---
Reasoning: The HVAC log has only `t` and `ambient` (likely a composite measure of hall conditions, maybe temperature/humidity). I need to merge it with historical records to see if `ambient` explains the time trend and the feedstock-outcome correlation. Let's get a larger sample of historical data and merge on `t` (assuming timestamps align). Since `t` is normalized across the era, we can merge approximately. I'll also compute correlations among ambient, feedstock, and outcome, and check if the feedstock-outcome correlation disappears after controlling for ambient.
--- celda que escribio ---
# Get more historical data and merge with HVAC
df_hist2 = env.observe('registros_linea', n=500)  # cost 250
df_hvac2 = env.observe('log_hvac', n=500)         # cost 25? Not specified, assume 0.5 per row? Actually not given; likely same as registros_linea cost_per_row? I'll guess 0.5.
merged = pd.merge(df_hist2, df_hvac2, on='t', how='i

...


 TURNO 8  |  presupuesto: 1975
--- razonamiento del agente ---
Reasoning: The experiment shows a huge effect of humidity on both feedstock reading and outcome: higher humidity (8 vs 2) drastically reduces both. This confirms that hall conditions (humidity) causally affect the process. Importantly, the feedstock reading is affected by humidity, meaning feedstock is not a pure measure of supplier grade but also sensitive to hall conditions. Thus, the historical correlation between feedstock and outcome is confounded by humidity. The remaining question: does supplier grade (feedstock_grade) have any direct causal effect on outcome, independent of hall conditions? We need to run an experiment where we vary feedstock_grade while holding humidity (and maybe other hall knobs) constant. Since we have limited budget, we can run a two-factor design: vary feedstock_grade at two levels (low=0, high=10) while fixing humidity at a middle value (say 5). We'll also [...]
--- celda que escribio ---
# 


 TURNO 9  |  presupuesto: 1335
--- razonamiento del agente ---
Reasoning: The factorial experiment reveals that feedstock_grade strongly affects feedstock reading (as expected: higher grade gives higher intake reading). However, outcome seems largely driven by humidity: at high humidity (8), outcome is low (~23) regardless of grade; at low humidity (2), outcome is high (~36-39) with a slight increase from grade 0 to 10. This suggests that supplier grade has a small positive effect on outcome only when humidity is favorable. The main effect of humidity is huge. The missing statsmodels is not crucial; we can compute simple differences. The key insight: the plant's proposed fix (switch to premium supplier) may have little effect if hall conditions remain poor (high humidity). Conversely, fixing hall conditions (lower humidity) could dramatically improve outcomes. We need to estimate the reject rate under two counterfactuals: (1) set feedstock_grade=10  [...]
--- celda que escribio ---
# 


 TURNO 10  |  presupuesto: 675
--- razonamiento del agente ---
Reasoning: The ambient distribution shows values roughly between 1 and 9, centered around 5. This matches the knob ranges (humidity 2-8). The error in the code is due to misusing `df.update`; I should assign columns differently. But more importantly, we need to see the effects of temp and line_speed. Let's fix the code and run the screening experiment. Given the budget is 675, we can afford this. Also, we need to synthesize a model soon. The key findings: humidity is a major driver; supplier grade has a small effect when humidity is low. We should also consider that feedstock reading is affected by both grade and humidity, and outcome is affected by humidity, grade, and possibly temp and line_speed. Let's run the screening correctly and then build a simple linear model.
--- celda que escribio ---
# Screening experiment for temp and line_speed, with fixed humidity and grade.
fixed = {'humidity': 5.0, 'feedstock_grade': 5.0}


 TURNO 11  |  presupuesto: 35
--- razonamiento del agente ---
Reasoning: The screening shows that temp and line_speed have relatively small effects on outcome (range ~28.5-30.2) compared to humidity's huge effect (~12.5 unit difference). Feedstock grade has a modest positive effect (~3.3) only when humidity is low; at high humidity, grade effect is negligible (maybe negative noise). This suggests that improving hall conditions (lower humidity) is more impactful than switching supplier, especially since historical ambient distribution is centered around 5 (moderate). The plant's folklore that supplier is the cause is wrong; the correlation is due to humidity affecting both feedstock reading and outcome. I now have enough to build a causal model: outcome is a function of humidity, feedstock_grade (with interaction), and small contributions from temp and line_speed. Feedstock reading is a function of humidity and feedstock_grade (and maybe temp/line_s [...]
--- celda que escribio ---
# B


>>> ENTREGO. El server puntua en silencio (el agente jamas ve esto):


### La nota y la disección

El server puntuó la entrega con el camino de producción completo (batería certificada, anclas,
funcionales, sandbox — cero LLM). Ahora abrimos el resultado: la nota, el código que entregó, y
la pregunta del millón — **¿compró el folklore o encontró la humedad?** — contestada comparando
su modelo contra la verdad en los dos counterfactuals que el cliente paga.

In [16]:
res = server_ep.result or {}
print(f"R = {res.get('R'):.4f}   (sin recorte: {res.get('R_unclipped'):.4f})")
print(f"tokens del episodio: {chat_ep.usage.total_tokens}  (~US$ {chat_ep.usage.total_tokens/1e6*0.6:.3f})")
print("\n--- el modelo que entrego ---\n")
print(res.get("code", "(no entrego)"))

R = 0.8986   (sin recorte: 0.8986)
tokens del episodio: 61607  (~US$ 0.037)

--- el modelo que entrego ---


import numpy as np
import pandas as pd

def model(regime, n, seed):
    rng = np.random.default_rng(seed)
    grade = regime.config.get('feedstock_grade', 5.0)
    humidity = regime.config.get('humidity', 5.0)
    temp = regime.config.get('temp', 5.0)
    line_speed = regime.config.get('line_speed', 5.0)
    
    # Outcome model coefficients from factorial experiment
    outcome_det = 40.25 - 2.09*humidity + 0.468*grade - 0.0675*humidity*grade
    # Residual variance estimated from humidity experiment (approx 4.0)
    outcome = outcome_det + rng.normal(0, 2.0, n)
    
    # Feedstock model coefficients from factorial experiment
    feedstock_det = 5.4666 - 0.8533*humidity + 1.021*grade - 0.0095*humidity*grade
    feedstock = feedstock_det + rng.normal(0, 1.0, n)
    
    return pd.DataFrame({'feedstock': feedstock, 'outcome': outcome})



In [17]:
# la diseccion: su modelo vs la VERDAD del caso, en las preguntas que el cliente paga
import importlib.util
spec = importlib.util.spec_from_file_location(
    "fs_world", ROOT / "cases" / "first_story_scarce_v0" / "world.py")
fs_world = importlib.util.module_from_spec(spec); spec.loader.exec_module(fs_world)

if res.get("code"):
    print(f"{'pregunta':42} {'VERDAD':>16} {'SU MODELO':>16}")
    with SandboxedSubmission(res["code"], ["feedstock", "outcome"], timeout_s=15.0) as sb:
        for nombre, cfg in [("historico (asi venia la planta)", {}),
                            ("do(proveedor premium): grade=10", {"feedstock_grade": 10.0}),
                            ("do(arreglar el hall): humidity=2", {"humidity": 2.0})]:
            ns = regime(cfg)
            verdad = fs_world.sample(ns, 4000, 42)["outcome"]
            suyo = sb.run(ns, 4000, 43)["outcome"]
            v = f"{verdad.mean():6.1f} ({(verdad < 25).mean():4.0%} rech)"
            s = f"{suyo.mean():6.1f} ({(suyo < 25).mean():4.0%} rech)"
            print(f"{nombre:42} {v:>16} {s:>16}")
    print("\nSi su modelo predice mejora con el proveedor premium, compro el folklore.")
    print("Si predice la mejora solo al arreglar el hall, encontro la historia verdadera.")

pregunta                                             VERDAD        SU MODELO


historico (asi venia la planta)              30.1 ( 20% rech)   30.4 (  0% rech)
do(proveedor premium): grade=10              30.2 ( 20% rech)   31.1 (  0% rech)
do(arreglar el hall): humidity=2             37.6 (  0% rech)   37.7 (  0% rech)

Si su modelo predice mejora con el proveedor premium, compro el folklore.
Si predice la mejora solo al arreglar el hall, encontro la historia verdadera.


## Cierre — dónde vive cada pieza

| Lo que viste | La pieza real |
|---|---|
| el mundo / la verdad | `cases/<caso>/world.py` (`sample(regime, n, seed)`) |
| regímenes y contratos | `wager/contracts.py` (`Regime`, `Battery`, `ExperimentDesign`…) |
| fuentes, canal, precios | `meta.json` del caso (`episode.observe_sources`, `channel`, `hidden_columns`) |
| el server del episodio | `wager/harness/world_server.py` + `build_world_server` |
| el kernel que usa el agente LLM | `wager/harness/kernel_proc.py` (el agente escribe celdas; `env` = este server) |
| la distancia | `wager/reward/distance.py` (`TruthSide`) |
| el scoring completo | `wager/reward/scorer.py` (batería + anclas + funcionales + MDL, cero-LLM) |
| el sandbox de submissions | `wager/reward/sandbox.py` |
| batería/rivales/certificados | `wager/factory/` (derivados de `meta.json`, nunca a mano) |
| el contrato de entrega v1 | `scripts/out/probe_0135/DELIVERY_SPEC_v1.md` |

Con la Parte 9 el recorrido queda **completamente end-to-end**: viste el mundo por dentro, las
vistas corrompidas, la presión del presupuesto, dos estilos de juicio y su precio, el contrato de
entrega, la nota — y al final un agente real jugando el examen entero mientras el server lo
puntúa a ciegas. Re-ejecutá la Parte 9 con otro `seed_offset` para ver otra partida (y la
varianza real entre corridas).